## Metrics

This notebooks demonstates how to work with metrics:
- usage of default aiice metrics
- custom metric implementation

In [1]:
from pprint import pprint
import torch
import numpy as np
torch.manual_seed(42)

# example batch of 4 grayscale images 64x64
y_true = torch.rand(4, 1, 64, 64)
y_pred = y_true + 0.1 * torch.randn(4, 1, 64, 64)  # noisy prediction

y_true.shape, y_pred.shape

(torch.Size([4, 1, 64, 64]), torch.Size([4, 1, 64, 64]))

In [2]:
from aiice.metrics import (
    mae, mse, rmse, psnr, bin_accuracy, ssim, Evaluator
)

### MAE

In [3]:
mae(y_true, y_pred)

0.0800759494304657

### MSE, RMSE

In [4]:
mse(y_true, y_pred), rmse(y_true, y_pred)

(0.010089104995131493, 0.10044454038143158)

### PSNR

In [5]:
psnr(y_true, y_pred)

19.960927963256836

### Bin Accuracy

In [6]:
bin_accuracy(y_true, y_pred, threshold=0.5)

0.9205322265625

### SSIM

In [7]:
ssim(y_true, y_pred)

0.9418593645095825

## Evaluator

`Evaluator` is necesarry for a batch-wise aggregation of different metrics. The result of one eval step is a dict of metrics per one step

In [8]:
evaluator = Evaluator()

In [9]:
step_metrics = evaluator.eval(y_true, y_pred)

pprint(step_metrics)

{'bin_accuracy': 0.9205322265625,
 'mae': 0.0800759494304657,
 'mse': 0.010089104995131493,
 'psnr': 19.960927963256836,
 'rmse': 0.10044454038143158,
 'ssim': 0.9418593645095825}


However you can eval several steps and get the final accumulative result with base statistics

In [10]:
for _ in range(5):
    y_pred_step = y_true + 0.1 * torch.randn_like(y_true)
    evaluator.eval(y_true, y_pred_step)

pprint(evaluator.report())

{'bin_accuracy': {'count': 6,
                  'last': 0.9183349609375,
                  'max': 0.92205810546875,
                  'mean': 0.9202982584635416,
                  'min': 0.9183349609375},
 'mae': {'count': 6,
         'last': 0.08009211719036102,
         'max': 0.08074222505092621,
         'mean': 0.08019860833883286,
         'min': 0.07981860637664795},
 'mse': {'count': 6,
         'last': 0.01005820743739605,
         'max': 0.010268930345773697,
         'mean': 0.010113622061908245,
         'min': 0.010042031295597553},
 'psnr': {'count': 6,
          'last': 19.9742488861084,
          'max': 19.981239318847656,
          'mean': 19.950518290201824,
          'min': 19.88420295715332},
 'rmse': {'count': 6,
          'last': 0.10029061138629913,
          'max': 0.10133573412895203,
          'mean': 0.10056575015187263,
          'min': 0.10020993649959564},
 'ssim': {'count': 6,
          'last': 0.9417504668235779,
          'max': 0.9421761631965637,
    

You can choose, which metrics you want to evaluate

In [11]:
evaluator_small = Evaluator(metrics=["mae", "ssim"])
evaluator_small.eval(y_true, y_pred)

pprint(evaluator_small.report())

{'mae': {'count': 1,
         'last': 0.0800759494304657,
         'max': 0.0800759494304657,
         'mean': 0.0800759494304657,
         'min': 0.0800759494304657},
 'ssim': {'count': 1,
          'last': 0.9418593645095825,
          'max': 0.9418593645095825,
          'mean': 0.9418593645095825,
          'min': 0.9418593645095825}}


Or you can evaluate metrics without any accumalition

In [12]:
evaluator_no_acc = Evaluator(accumulate=False)

for _ in range(3):
    y_pred_step = y_true + 0.2 * torch.randn_like(y_true)
    evaluator_no_acc.eval(y_true, y_pred_step)

# contains only last step
pprint(evaluator_no_acc.report())

{'bin_accuracy': {'count': 1,
                  'last': 0.84173583984375,
                  'max': 0.84173583984375,
                  'mean': 0.84173583984375,
                  'min': 0.84173583984375},
 'mae': {'count': 1,
         'last': 0.16021728515625,
         'max': 0.16021728515625,
         'mean': 0.16021728515625,
         'min': 0.16021728515625},
 'mse': {'count': 1,
         'last': 0.04005054756999016,
         'max': 0.04005054756999016,
         'mean': 0.04005054756999016,
         'min': 0.04005054756999016},
 'psnr': {'count': 1,
          'last': 13.973368644714355,
          'max': 13.973368644714355,
          'mean': 13.973368644714355,
          'min': 13.973368644714355},
 'rmse': {'count': 1,
          'last': 0.20012633502483368,
          'max': 0.20012633502483368,
          'mean': 0.20012633502483368,
          'min': 0.20012633502483368},
 'ssim': {'count': 1,
          'last': 0.7997472286224365,
          'max': 0.7997472286224365,
          'mean'

Also, you can set your own metrics, which should follow a `MetricFn` type:
```python
MetricFn = Callable[[Sequence, Sequence], float]
```

In [13]:
def sum_error(y_true, y_pred):
    y_true = torch.as_tensor(y_true, dtype=torch.float32)
    y_pred = torch.as_tensor(y_pred, dtype=torch.float32)
    return float((y_pred - y_true).sum())

In [14]:
evaluator = Evaluator(
    metrics={
        "mae": mae,              # built-in
        "sum_error": sum_error,  # custom
    }
)

In [15]:
y_true = torch.tensor([0.0, 1.0, 2.0])
y_pred = torch.tensor([0.1, 0.9, 2.5])

evaluator.eval(y_true, y_pred)
pprint(evaluator.report())

{'mae': {'count': 1,
         'last': 0.23333334922790527,
         'max': 0.23333334922790527,
         'mean': 0.23333334922790527,
         'min': 0.23333334922790527},
 'sum_error': {'count': 1,
               'last': 0.4999999701976776,
               'max': 0.4999999701976776,
               'mean': 0.4999999701976776,
               'min': 0.4999999701976776}}
